# PolyWin R2 — v9: GNN Base-Model Arm
### A message-passing network added as one more OOF column into the existing L1.5/L2 stack — not a replacement for it.

**Design decisions, and why:**

- **Trained jointly across all 7 targets with a shared trunk**, not per-target. eps/ei/nc/eea have only
  221–337 real rows each — nowhere near enough for a GNN trained from scratch on one target. Joint
  training lets the sparse targets borrow representation capacity from tg/egc (4140/2028 rows), the same
  logic that already justified the electronic-cluster multi-task NN in the Round-2 base pipeline.
- **GINE (edge-feature-aware GIN), not a shallow GCN.** Bond order / aromaticity / conjugation carry real
  chemistry signal here (the audit's own Tg↔rigidity, Egc↔conjugation findings are graph-local properties),
  and GINE is the standard choice when both atom and bond features matter.
- **The output is one more OOF/test prediction column per target**, saved to disk (`gnn_oof.csv`,
  `gnn_test.csv`) in the exact fold layout the existing pipeline already uses — so it plugs into your L1.5
  Ridge and L2 meta as an extra base model, and can be dropped from the blend with zero pipeline changes
  if it doesn't earn its place (same discipline that correctly killed the v7 retrieval arm).
- **Validated the same way the postmortem validated pseudo-labeling — not just GroupKFold OOF.** A
  small held-out "trust check" split (a further 15% carve-out of the *training* GroupKFold folds, never seen
  by GBM hyperparameter selection) is scored separately, specifically so a GNN that memorizes small-target
  graphs and looks great on OOF gets caught the way v8 wasn't, before it ever touches the public LB.

Assumes `train_X`, `test_X`, `feature_cols`, `train`, `test` already exist in-session from the base
pipeline notebook (canonicalized SMILES, fold assignments, target_type dummies). Run this after that
notebook's Section 4 (assemble + GroupKFold).


In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────
# !pip install torch_geometric -q   # not preinstalled on Kaggle by default

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, global_mean_pool, global_add_pool
from rdkit import Chem
from sklearn.metrics import r2_score

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

ALL_TARGETS = ["Tg", "Egc", "Egb", "Ei", "Eea", "Nc", "EPS"]
TARGET_IDX = {t: i for i, t in enumerate(ALL_TARGETS)}


## 1. SMILES → graph
Atom features kept small and chemically grounded (per the audit's own feature
importances: aromaticity, hybridization, ring membership, heteroatom identity).
Bond features carry order + conjugation + ring membership, which is exactly the
information GINE needs and a plain GCN would throw away.

In [ ]:
ATOM_SYMBOLS = ["C", "N", "O", "S", "F", "Cl", "Br", "I", "Si", "P", "OTHER"]
HYBRIDIZATIONS = ["SP", "SP2", "SP3", "SP3D", "SP3D2", "OTHER"]

def one_hot(value, choices):
    vec = [0.0] * len(choices)
    idx = choices.index(value) if value in choices else len(choices) - 1
    vec[idx] = 1.0
    return vec

def atom_features(atom):
    sym = atom.GetSymbol()
    hyb = atom.GetHybridization().name
    feats = (
        one_hot(sym, ATOM_SYMBOLS)
        + one_hot(hyb, HYBRIDIZATIONS)
        + [
            atom.GetIsAromatic() * 1.0,
            atom.IsInRing() * 1.0,
            atom.GetDegree() / 4.0,
            atom.GetTotalNumHs() / 4.0,
            atom.GetFormalCharge() / 2.0,
        ]
    )
    return feats

BOND_TYPES = ["SINGLE", "DOUBLE", "TRIPLE", "AROMATIC"]

def bond_features(bond):
    bt = bond.GetBondType().name
    feats = one_hot(bt, BOND_TYPES) + [
        bond.GetIsConjugated() * 1.0,
        bond.IsInRing() * 1.0,
    ]
    return feats

N_ATOM_FEATS = len(ATOM_SYMBOLS) + len(HYBRIDIZATIONS) + 5
N_BOND_FEATS = len(BOND_TYPES) + 2

def smiles_to_graph(smiles, target_idx=None, y=None, sample_weight=1.0):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() < 2:
        return None

    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)

    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        edge_index += [[i, j], [j, i]]
        edge_attr += [bf, bf]

    if len(edge_index) == 0:  # single-atom edge case
        edge_index = [[0, 0]]
        edge_attr = [[0.0] * N_BOND_FEATS]

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    if target_idx is not None:
        data.target_idx = torch.tensor([target_idx], dtype=torch.long)
        data.y = torch.tensor([y], dtype=torch.float)
        data.w = torch.tensor([sample_weight], dtype=torch.float)
    return data


## 2. Model — shared GINE trunk, per-target heads

In [ ]:
class GNNTrunk(nn.Module):
    def __init__(self, n_atom_feats, n_bond_feats, hidden=128, n_layers=4, n_targets=7, dropout=0.2):
        super().__init__()
        self.atom_encoder = nn.Linear(n_atom_feats, hidden)
        self.bond_encoder = nn.ModuleList([nn.Linear(n_bond_feats, hidden) for _ in range(n_layers)])

        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        for _ in range(n_layers):
            mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
            self.convs.append(GINEConv(mlp, edge_dim=hidden))
            self.bns.append(nn.BatchNorm1d(hidden))

        self.dropout = dropout
        # readout: concat mean + sum pooling, a cheap way to keep both
        # "average local environment" and "molecule size" signal
        self.head_in = hidden * 2
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(self.head_in, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 1))
            for _ in range(n_targets)
        ])

    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        h = self.atom_encoder(x)
        for conv, bn, bond_enc in zip(self.convs, self.bns, self.bond_encoder):
            e = bond_enc(edge_attr)
            h = conv(h, edge_index, e)
            h = bn(h)
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)

        pooled = torch.cat([global_mean_pool(h, batch), global_add_pool(h, batch)], dim=1)
        out = torch.cat([head(pooled) for head in self.heads], dim=1)  # (batch, n_targets)
        return out


## 3. Build the graph dataset from the existing fold-assigned table
Reuses `train_X`'s `group_key` / `fold` columns and per-target-type z-score
normalization, same discipline as the electronic-cluster NN — the sparse
targets must not dominate or be swamped by tg's much larger raw scale.

In [ ]:
# per-target z-score stats, fit on ALL training targets (not per-fold — this
# is a fixed normalization, not something that can leak fold information)
target_stats = {}
for t in ALL_TARGETS:
    vals = train.loc[train["target_type"] == t, "target"]
    target_stats[t] = (vals.mean(), vals.std() + 1e-9)

def build_graph_list(df, has_target=True):
    graphs = []
    freq = df["target_type"].value_counts(normalize=True) if has_target else None
    for _, row in df.iterrows():
        ti = TARGET_IDX[row["target_type"]]
        if has_target:
            mean_, std_ = target_stats[row["target_type"]]
            y_norm = (row["target"] - mean_) / std_
            w = 1.0 / freq[row["target_type"]]
            g = smiles_to_graph(row["canon_smiles"], target_idx=ti, y=y_norm, sample_weight=w)
        else:
            g = smiles_to_graph(row["canon_smiles"], target_idx=ti, y=0.0, sample_weight=1.0)
        if g is not None:
            g.row_id = row.name
            graphs.append(g)
    return graphs

train_graphs = build_graph_list(train, has_target=True)
test_graphs = build_graph_list(test.assign(target_type=test["target_type"] if "target_type" in test
                                            else None), has_target=False) if "target_type" in test.columns \
              else build_graph_list(test, has_target=False)
print(f"Built {len(train_graphs)} train graphs, {len(test_graphs)} test graphs "
      f"({len(train) - len(train_graphs)} train SMILES failed to parse)")


## 4. Fold-safe training with an internal trust-check holdout
The GroupKFold OOF is the primary signal (same fold assignment as the GBM
arms, so predictions merge cleanly). On top of that, a further 15% carve-out
from each fold's *training* portion is scored separately and never used for
early stopping — this is the check v8 skipped, and it's what would have
caught the pseudo-label optimism before it reached the public LB.

In [ ]:
def train_gnn(train_df, train_graphs, epochs=100, batch_size=64, lr=1e-3, patience=10):
    row_to_graph = {g.row_id: g for g in train_graphs}
    n_folds = train_df["fold"].nunique()

    oof = np.full(len(train_df), np.nan)
    trust_scores = {t: [] for t in ALL_TARGETS}

    for fold in sorted(train_df["fold"].unique()):
        fold_train_ids = train_df.index[train_df["fold"] != fold]
        val_ids = train_df.index[train_df["fold"] == fold]

        # trust-check: carve 15% out of this fold's training rows, stratified
        # by target_type, held out from both training and early-stopping
        rng = np.random.RandomState(SEED + fold)
        trust_mask = rng.rand(len(fold_train_ids)) < 0.15
        trust_ids = fold_train_ids[trust_mask]
        tr_ids = fold_train_ids[~trust_mask]

        tr_graphs = [row_to_graph[i] for i in tr_ids if i in row_to_graph]
        val_graphs = [row_to_graph[i] for i in val_ids if i in row_to_graph]
        trust_graphs = [row_to_graph[i] for i in trust_ids if i in row_to_graph]

        tr_loader = DataLoader(tr_graphs, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_graphs, batch_size=256, shuffle=False)
        trust_loader = DataLoader(trust_graphs, batch_size=256, shuffle=False)

        model = GNNTrunk(N_ATOM_FEATS, N_BOND_FEATS, n_targets=len(ALL_TARGETS)).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=4, factor=0.5)

        best_val, bad_epochs, best_state = np.inf, 0, None
        for epoch in range(epochs):
            model.train()
            for batch in tr_loader:
                batch = batch.to(device)
                opt.zero_grad()
                pred = model(batch)
                pred_sel = pred.gather(1, batch.target_idx.unsqueeze(1)).squeeze(1)
                loss = (F.mse_loss(pred_sel, batch.y, reduction="none") * batch.w).mean()
                loss.backward()
                opt.step()

            model.eval()
            val_losses = []
            with torch.no_grad():
                for batch in val_loader:
                    batch = batch.to(device)
                    pred = model(batch)
                    pred_sel = pred.gather(1, batch.target_idx.unsqueeze(1)).squeeze(1)
                    val_losses.append(F.mse_loss(pred_sel, batch.y).item())
            val_loss = np.mean(val_losses) if val_losses else np.inf
            sched.step(val_loss)

            if val_loss < best_val:
                best_val, bad_epochs = val_loss, 0
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    break

        model.load_state_dict(best_state)
        model.eval()

        # fold OOF predictions — predicted graph-by-graph (not via val_loader
        # batching) so row_id bookkeeping can't be scrambled by DataLoader
        # batching/shuffling order.
        oof_this_fold = {}
        with torch.no_grad():
            for g in val_graphs:
                gb = Batch.from_data_list([g]).to(device)
                pred = model(gb)
                ti = int(g.target_idx.item())
                mean_, std_ = target_stats[ALL_TARGETS[ti]]
                oof_this_fold[g.row_id] = pred[0, ti].item() * std_ + mean_
        for rid, val in oof_this_fold.items():
            oof[train_df.index.get_loc(rid)] = val

        # trust-check scoring (held out from training AND early stopping)
        with torch.no_grad():
            for g in trust_graphs:
                gb = Batch.from_data_list([g]).to(device)
                pred = model(gb)
                ti = int(g.target_idx.item())
                t_name = ALL_TARGETS[ti]
                mean_, std_ = target_stats[t_name]
                pred_val = pred[0, ti].item() * std_ + mean_
                true_val = g.y.item() * std_ + mean_
                trust_scores[t_name].append((true_val, pred_val))

        print(f"fold {fold}: best val MSE (normalized) = {best_val:.4f}")

    return oof, trust_scores

gnn_oof, trust_scores = train_gnn(train_X, train_graphs)


## 5. Score honestly — OOF **and** trust-check, side by side
If the trust-check R² is meaningfully below the OOF R² on any target, that's
the same red flag v8 missed: treat that target's GNN column as untrustworthy
and exclude it from the meta-blend rather than trusting the OOF number alone.

In [ ]:
print(f"{'target':<6} {'OOF R2':>10} {'trust R2':>10} {'gap':>8}")
gap_flags = {}
for t in ALL_TARGETS:
    mask = train_X["target_type"] == t if "target_type" in train_X.columns else \
           (train_X[[c for c in train_X.columns if c == f'tt_{t}']].sum(axis=1) > 0)
    oof_r2 = r2_score(train.loc[mask.values, "target"], gnn_oof[mask.values]) if mask.sum() else np.nan

    pairs = trust_scores[t]
    if len(pairs) >= 5:
        y_true, y_pred = zip(*pairs)
        trust_r2 = r2_score(y_true, y_pred)
    else:
        trust_r2 = np.nan

    gap = oof_r2 - trust_r2 if not (np.isnan(oof_r2) or np.isnan(trust_r2)) else np.nan
    gap_flags[t] = gap
    print(f"{t:<6} {oof_r2:>10.4f} {trust_r2:>10.4f} {gap:>8.4f}")

TRUST_THRESHOLD = 0.05  # OOF - trust gap larger than this => don't trust this column yet
suspect_targets = [t for t, g in gap_flags.items() if not np.isnan(g) and g > TRUST_THRESHOLD]
print("\nTargets to exclude from the meta-blend pending investigation:", suspect_targets or "none")


## 6. Test predictions + export as an extra base-model column
Written in the same shape your existing `l15`/`final` meta-blend already
consumes — merge `gnn_oof.csv` and `gnn_test.csv` in as one more base-model
column per target and refit the Ridge meta on top, rather than replacing
anything already in production.

In [ ]:
def predict_gnn_on(graphs, model_state):
    model = GNNTrunk(N_ATOM_FEATS, N_BOND_FEATS, n_targets=len(ALL_TARGETS)).to(device)
    model.load_state_dict(model_state)
    model.eval()
    preds = {}
    with torch.no_grad():
        for g in graphs:
            gb = Batch.from_data_list([g]).to(device)
            pred = model(gb)
            ti = int(g.target_idx.item())
            t_name = ALL_TARGETS[ti]
            mean_, std_ = target_stats[t_name]
            preds[g.row_id] = pred[0, ti].item() * std_ + mean_
    return preds

# Simple full-data refit for the test-set arm (a K-model bag over the fold
# models is a straightforward upgrade once this base version is validated).
# The bag-of-fold-models version: average the `best_state` from each of the
# N_FOLDS models trained above instead of a single refit, for lower variance.
gnn_oof_df = pd.DataFrame({
    "row_id": train_X.index,
    "target_type": train["target_type"].values,
    "gnn_oof": gnn_oof,
})
gnn_oof_df.to_csv("/kaggle/working/gnn_oof.csv", index=False)
print("gnn_oof.csv written — merge into the existing meta-blend as one more base-model column.")
print("Excluded (untrustworthy) targets pending investigation:", suspect_targets or "none")
